# Ejecución, monitoreo e incertidumbre

## Conceptos clave

### 1. Monitoreo de ejecución
- Verificar si el plan se está cumpliendo.
- Detectar desviaciones del estado esperado.

### 2. Replanificación
- Generar un nuevo plan cuando el actual falla.
- Puede ser:
  - Completa (desde cero)
  - Parcial (reparación del plan)

### 3. Incertidumbre
- Acciones con efectos no deterministas.
- Información incompleta del entorno.

### 4. Planificación reactiva vs deliberativa

| Tipo           | Característica |
|----------------|--------------|
| Reactiva       | Respuesta inmediata (reglas) |
| Deliberativa   | Planifica antes de actuar |

Hoy vamos a SIMULAR estos conceptos.

In [ ]:
# Carga de librerías
import random

In [ ]:
class Environment:
    def __init__(self, size=5, obstacle_prob=0.2):
        self.size = size
        self.obstacle_prob = obstacle_prob
        self.reset()

    def reset(self):
        self.grid = [[0 for _ in range(self.size)] for _ in range(self.size)]
        self.agent = (0, 0)
        self.goal = (self.size-1, self.size-1)
        self.generate_obstacles()

    def generate_obstacles(self):
        for i in range(self.size):
            for j in range(self.size):
                if (i, j) not in [self.agent, self.goal]:
                    if random.random() < self.obstacle_prob:
                        self.grid[i][j] = 1

    def is_free(self, pos):
        x, y = pos
        return 0 <= x < self.size and 0 <= y < self.size and self.grid[x][y] == 0

    def step(self, action):
        x, y = self.agent
        moves = {
            "UP": (x-1, y),
            "DOWN": (x+1, y),
            "LEFT": (x, y-1),
            "RIGHT": (x, y+1)
        }

        if action in moves and self.is_free(moves[action]):
            self.agent = moves[action]

        return self.agent == self.goal

## Planificador deliberativo (BFS simple)

In [ ]:
from collections import deque

def bfs_plan(env):
    start = env.agent
    goal = env.goal

    queue = deque([(start, [])])
    visited = set()

    while queue:
        state, path = queue.popleft()

        if state == goal:
            return path

        if state in visited:
            continue

        visited.add(state)

        for action in ["UP", "DOWN", "LEFT", "RIGHT"]:
            x, y = state
            moves = {
                "UP": (x-1, y),
                "DOWN": (x+1, y),
                "LEFT": (x, y-1),
                "RIGHT": (x, y+1)
            }

            next_state = moves[action]
            if env.is_free(next_state):
                queue.append((next_state, path + [action]))

    return None

## Monitoreo de ejecución + Replanificación

In [ ]:
def execute_with_monitoring(env):
    plan = bfs_plan(env)

    if not plan:
        print("No se encontró plan inicial")
        return

    print("Plan inicial:", plan)

    for step, action in enumerate(plan):
        # Introducimos incertidumbre dinámica
        if random.random() < 0.3:
            env.generate_obstacles()
            print(f"Cambio en entorno en paso {step}")

        # Monitoreo: verificar si la acción sigue siendo válida
        prev_pos = env.agent
        env.step(action)

        if env.agent == prev_pos:
            print("Acción falló. Replanificando...")
            new_plan = bfs_plan(env)

            if not new_plan:
                print("No se pudo replanificar")
                return

            print("Nuevo plan:", new_plan)
            return execute_with_monitoring(env)

        print(f"Paso {step}: {action} -> {env.agent}")

        if env.agent == env.goal:
            print("Objetivo alcanzado")
            return

# Ejecutar simulación

In [ ]:
env = Environment(size=6, obstacle_prob=0.2)

execute_with_monitoring(env)

# Planificación reactiva (sin plan global)

In [ ]:
def reactive_agent(env):
    print("Agente reactivo en ejecución")

    for step in range(50):
        x, y = env.agent
        gx, gy = env.goal

        # Estrategia greedy simple
        actions = []
        if gx > x: actions.append("DOWN")
        if gx < x: actions.append("UP")
        if gy > y: actions.append("RIGHT")
        if gy < y: actions.append("LEFT")

        random.shuffle(actions)

        moved = False
        for action in actions:
            prev = env.agent
            env.step(action)
            if env.agent != prev:
                moved = True
                break

        if not moved:
            print("⚠️ Bloqueado, moviendo aleatoriamente")
            env.step(random.choice(["UP","DOWN","LEFT","RIGHT"]))

        print(f"Paso {step}: {env.agent}")

        if env.agent == env.goal:
            print("Objetivo alcanzado (reactivo)")
            return

# Comparación experimental

In [ ]:
def compare_agents():
    print("==== DELIBERATIVO ====")
    env1 = Environment()
    execute_with_monitoring(env1)

    print("\n==== REACTIVO ====")
    env2 = Environment()
    reactive_agent(env2)

compare_agents()

# Visualización tipo grilla

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time
from matplotlib import colors

class Visualizer:
    def __init__(self, env):
        self.env = env
        self.history = []

        # Mapa de colores
        self.cmap = colors.ListedColormap([
            'white',    # 0 libre
            'black',    # 1 obstáculo
            'blue',     # 2 agente
            'green',    # 3 meta
            'lightblue' # 4 camino recorrido
        ])

    def build_grid(self):
        grid = np.array(self.env.grid).copy()

        # Marcar historial
        for (x, y) in self.history:
            if grid[x][y] == 0:
                grid[x][y] = 4

        # Agente y meta
        ax, ay = self.env.agent
        gx, gy = self.env.goal

        grid[gx][gy] = 3
        grid[ax][ay] = 2

        return grid

    def plot(self, step=None, title_extra=""):
        grid = self.build_grid()

        plt.figure(figsize=(6,6))
        plt.imshow(grid, cmap=self.cmap)

        # Dibujar grilla
        plt.grid(True)
        plt.xticks(range(self.env.size))
        plt.yticks(range(self.env.size))

        # Coordenadas en cada celda
        for i in range(self.env.size):
            for j in range(self.env.size):
                plt.text(j, i, f"({i},{j})",
                         ha='center', va='center',
                         color='gray', fontsize=8)

        title = "Estado del entorno"
        if step is not None:
            title += f" | Paso {step}"
        if title_extra:
            title += f" | {title_extra}"

        plt.title(title)

        # Leyenda manual
        import matplotlib.patches as mpatches
        legend = [
            mpatches.Patch(color='white', label='Libre'),
            mpatches.Patch(color='black', label='Obstáculo'),
            mpatches.Patch(color='blue', label='Agente'),
            mpatches.Patch(color='green', label='Meta'),
            mpatches.Patch(color='lightblue', label='Recorrido')
        ]
        plt.legend(handles=legend, bbox_to_anchor=(1.05, 1), loc='upper left')

        plt.show()

    def update(self):
        self.history.append(self.env.agent)

    def animate_execution(self, plan, delay=0.7):
        print("Animación del plan...")

        for step, action in enumerate(plan):
            self.update()
            self.plot(step, f"Acción: {action}")
            time.sleep(delay)

            # Simular ejecución real
            self.env.step(action)

        self.update()
        self.plot(step=len(plan), title_extra="FIN")

# Uso integrado con el planner

In [ ]:
env = Environment(size=6, obstacle_prob=0.2)
viz = Visualizer(env)

plan = bfs_plan(env)

if plan:
    print("Plan encontrado:", plan)
    viz.animate_execution(plan)
else:
    print("No hay plan")
    viz.plot()

# Versión con incertidumbre

Esto muestra claramente el concepto de monitoreo + cambio del entorno:

In [ ]:
import random
from collections import deque

class Environment:
    def __init__(self, size=6, obstacle_prob=0.2):
        self.size = size
        self.obstacle_prob = obstacle_prob
        self.agent = (0, 0)
        self.goal = (size-1, size-1)
        self.reset()

    def reset(self):
        while True:
            self.grid = [[0 for _ in range(self.size)] for _ in range(self.size)]

            for i in range(self.size):
                for j in range(self.size):
                    if (i, j) not in [self.agent, self.goal]:
                        if random.random() < self.obstacle_prob:
                            self.grid[i][j] = 1

            if self._has_path():
                break

    def is_free(self, pos):
        x, y = pos
        return 0 <= x < self.size and 0 <= y < self.size and self.grid[x][y] == 0

    def neighbors(self, pos):
        x, y = pos
        moves = [(x-1,y),(x+1,y),(x,y-1),(x,y+1)]
        return [m for m in moves if self.is_free(m)]

    def step(self, action):
        x, y = self.agent
        moves = {
            "UP": (x-1,y),
            "DOWN": (x+1,y),
            "LEFT": (x,y-1),
            "RIGHT": (x,y+1)
        }

        if action in moves and self.is_free(moves[action]):
            self.agent = moves[action]

        return self.agent == self.goal

    #INCERTIDUMBRE LOCAL
    def local_disturbance(self, prob=0.2, radius=2):
        ax, ay = self.agent

        for i in range(self.size):
            for j in range(self.size):
                if abs(i-ax) + abs(j-ay) <= radius:
                    if (i, j) not in [self.agent, self.goal]:
                        if random.random() < prob:
                            self.grid[i][j] = 1 - self.grid[i][j]

    def _has_path(self):
        queue = deque([self.agent])
        visited = set()

        while queue:
            current = queue.popleft()
            if current == self.goal:
                return True

            if current in visited:
                continue

            visited.add(current)

            for n in self.neighbors(current):
                queue.append(n)

        return False

In [ ]:
# Planner BFS
def bfs_plan(env):
    start = env.agent
    goal = env.goal

    queue = deque([(start, [])])
    visited = set()

    while queue:
        state, path = queue.popleft()

        if state == goal:
            return path

        if state in visited:
            continue

        visited.add(state)

        x, y = state
        moves = {
            "UP": (x-1,y),
            "DOWN": (x+1,y),
            "LEFT": (x,y-1),
            "RIGHT": (x,y+1)
        }

        for action, next_state in moves.items():
            if env.is_free(next_state):
                queue.append((next_state, path + [action]))

    return None

In [ ]:
# Visualización
import matplotlib.pyplot as plt
import numpy as np

class Visualizer:
    def __init__(self, env):
        self.env = env
        self.history = []

    def update(self):
        self.history.append(self.env.agent)

    def plot(self, step=0, title=""):
        grid = np.array(self.env.grid).copy()

        for (x,y) in self.history:
            if grid[x][y] == 0:
                grid[x][y] = 4

        ax, ay = self.env.agent
        gx, gy = self.env.goal

        grid[gx][gy] = 3
        grid[ax][ay] = 2

        plt.figure(figsize=(5,5))
        plt.imshow(grid)
        plt.title(f"Paso {step} | {title}")
        plt.grid(True)
        plt.show()

In [ ]:
# Ejecución
import time

def run_uncertainty_demo(size=6, obstacle_prob=0.2):
    print("=== DEMO COMPLETA ===\n")

    env = Environment(size=size, obstacle_prob=obstacle_prob)
    viz = Visualizer(env)

    plan = bfs_plan(env)

    if not plan:
        print("No hay plan inicial")
        return

    print("Plan inicial:", plan)
    viz.plot(0, "Inicio")

    step = 0

    while True:
        for action in plan:
            viz.update()

            # incertidumbre LOCAL
            if random.random() < 0.3:
                env.local_disturbance(prob=0.2, radius=2)
                viz.plot(step, "cambio local")
            else:
                viz.plot(step, f"Acción: {action}")

            prev = env.agent
            env.step(action)

            # 🔍 monitoreo
            if env.agent == prev:
                print("Acción falló → Replanificando")

                plan = bfs_plan(env)

                if not plan:
                    print("Sin solución")
                    return

                print("Nuevo plan:", plan)
                break

            step += 1

            if env.agent == env.goal:
                viz.update()
                viz.plot(step, "Objetivo alcanzado")
                print("✅ Éxito")
                return

        else:
            print("Plan agotado → Replanificar")
            plan = bfs_plan(env)

            if not plan:
                print("Sin solución final")
                return

In [ ]:
run_uncertainty_demo()